In [1]:
# ==========================================================
# Histology MIL with mask guided tiling (cached tiles) + Phikon (owkin/phikon)
# Fixed seed = 42
# Paths unchanged (same BASE_PATH structure as your previous code)
# ==========================================================

import os
import cv2
import gc
import math
import shutil
import random
import warnings
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from PIL import Image

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision import transforms

from transformers import ViTModel

warnings.filterwarnings("ignore")

# ---------------------------
# 1) CONFIG
# ---------------------------
class Config:
    SEED = 42

    # preprocessing output size (square) and tile size for ViT
    SAVE_SIZE = 768
    TILE_SIZE = 224

    # tiles to sample per slide
    MAX_TILES_PER_SLIDE = 80   # extracted once per slide
    K_TRAIN = 24               # sampled per epoch per slide
    K_VAL = 48                 # used for validation/inference

    MIN_MASK_FRAC = 0.01       # keep tiles with at least 1% mask
    MAX_TRIES = 400

    BATCH_SIZE = 4             # bags are heavier than single images
    EPOCHS = 25
    N_FOLDS = 3

    # optimization
    LR_HEAD = 3e-4
    LR_BACKBONE = 1e-5
    WEIGHT_DECAY = 1e-2
    LABEL_SMOOTH = 0.10

    FREEZE_EPOCHS = 3

    NUM_WORKERS = 4
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # HuggingFace backbone (open, not gated)
    BACKBONE = "owkin/phikon"  # ViT encoder

    # Paths (same structure as before)
    BASE_PATH = "/kaggle/input/dataset/"
    TRAIN_ROOT = os.path.join(BASE_PATH, "train_data_cleaned")
    if not os.path.exists(TRAIN_ROOT):
        TRAIN_ROOT = os.path.join(BASE_PATH, "train_data")

    TRAIN_IMG_DIR = os.path.join(TRAIN_ROOT, "images")
    TRAIN_MASK_DIR = os.path.join(TRAIN_ROOT, "masks")
    TRAIN_LABELS_CSV = os.path.join(TRAIN_ROOT, "train_labels.csv")

    TEST_ROOT = os.path.join(BASE_PATH, "test_data")
    TEST_IMG_DIR = os.path.join(TEST_ROOT, "images")
    TEST_MASK_DIR = os.path.join(TEST_ROOT, "masks")

    # Preprocessed full images and masks
    FAST_TRAIN_IMG = "/kaggle/working/fast_train/images"
    FAST_TRAIN_MASK = "/kaggle/working/fast_train/masks"
    FAST_TEST_IMG = "/kaggle/working/fast_test/images"
    FAST_TEST_MASK = "/kaggle/working/fast_test/masks"

    # Cached tiles
    TILES_TRAIN_DIR = "/kaggle/working/tiles_train"
    TILES_TEST_DIR = "/kaggle/working/tiles_test"

# ---------------------------
# 2) SEEDING
# ---------------------------
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

def seed_worker(worker_id):
    worker_seed = (Config.SEED + worker_id) % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

seed_everything(Config.SEED)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

# Clean previous outputs
for p in [
    "/kaggle/working/fast_train", "/kaggle/working/fast_test",
    Config.TILES_TRAIN_DIR, Config.TILES_TEST_DIR
]:
    if os.path.exists(p):
        shutil.rmtree(p)

# ---------------------------
# 3) PREPROCESS: REINHARD + ROI CROP (save RGB + mask)
# ---------------------------
class ReinhardNormalizer:
    def __init__(self):
        self.target_mean = np.array([148.60, 169.30, 105.47], dtype=np.float32)
        self.target_std  = np.array([41.56, 11.23, 14.39], dtype=np.float32)

    def __call__(self, img_rgb):
        try:
            img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
            img_mean = np.mean(img_lab, axis=(0, 1))
            img_std  = np.std(img_lab, axis=(0, 1)) + 1e-5
            img_lab = (img_lab - img_mean) / img_std * self.target_std + self.target_mean
            img_lab = np.clip(img_lab, 0, 255).astype(np.uint8)
            return cv2.cvtColor(img_lab, cv2.COLOR_LAB2RGB)
        except Exception:
            return img_rgb

def find_image_file(base_dir, filename_base):
    if not isinstance(filename_base, str):
        return None
    fid = filename_base.split(".")[0]
    for ext in [".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"]:
        p = os.path.join(base_dir, fid + ext)
        if os.path.exists(p):
            return p
    return None

def smart_crop_dual(img, mask, context=1.2):
    if mask is None or np.sum(mask) == 0:
        h, w = img.shape[:2]
        s = min(h, w)
        y = (h - s) // 2
        x = (w - s) // 2
        return img[y:y+s, x:x+s], mask[y:y+s, x:x+s] if mask is not None else np.zeros((s, s), dtype=np.uint8)

    ys, xs = np.where(mask > 0)
    y0, y1 = int(ys.min()), int(ys.max())
    x0, x1 = int(xs.min()), int(xs.max())

    h_roi = y1 - y0
    w_roi = x1 - x0
    side = int(max(h_roi, w_roi) * context)
    side = max(side, 32)

    cy = y0 + h_roi // 2
    cx = x0 + w_roi // 2

    y_start = max(0, cy - side // 2)
    y_end   = min(img.shape[0], cy + side // 2)
    x_start = max(0, cx - side // 2)
    x_end   = min(img.shape[1], cx + side // 2)

    crop_img = img[y_start:y_end, x_start:x_end]
    crop_msk = mask[y_start:y_end, x_start:x_end]

    pad_h = side - crop_img.shape[0]
    pad_w = side - crop_img.shape[1]
    if pad_h > 0 or pad_w > 0:
        crop_img = cv2.copyMakeBorder(
            crop_img,
            pad_h // 2, pad_h - pad_h // 2,
            pad_w // 2, pad_w - pad_w // 2,
            borderType=cv2.BORDER_CONSTANT,
            value=[255, 255, 255]
        )
        crop_msk = cv2.copyMakeBorder(
            crop_msk,
            pad_h // 2, pad_h - pad_h // 2,
            pad_w // 2, pad_w - pad_w // 2,
            borderType=cv2.BORDER_CONSTANT,
            value=0
        )

    return crop_img, crop_msk

def preprocess_rgb_and_mask(df, img_dir, mask_dir, out_img_dir, out_mask_dir, is_test=False):
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_mask_dir, exist_ok=True)

    normalizer = ReinhardNormalizer()
    valid_files = []

    print(f"Processing to {out_img_dir} ...")
    for _, row in tqdm(df.iterrows(), total=len(df)):
        if is_test:
            fname = row["filename"]
        else:
            fname = row.get("image_name", row.get("sample_index", row.get("filename")))
            if not isinstance(fname, str):
                fname = row[0]

        img_p = find_image_file(img_dir, fname)
        if img_p is None:
            continue
        base = fname.split(".")[0]

        if "img_" in base:
            mask_base = base.replace("img_", "mask_")
        elif "img" in base:
            mask_base = base.replace("img", "mask")
        else:
            mask_base = "mask_" + base

        mask_p = find_image_file(mask_dir, mask_base)

        out_rgb_p = os.path.join(out_img_dir, base + ".png")
        out_msk_p = os.path.join(out_mask_dir, base + ".png")

        if os.path.exists(out_rgb_p) and os.path.exists(out_msk_p):
            valid_files.append(base + ".png")
            continue

        try:
            img = cv2.imread(img_p)
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            if mask_p is not None:
                mask = cv2.imread(mask_p, cv2.IMREAD_GRAYSCALE)
                if mask is None:
                    mask = np.zeros(img.shape[:2], dtype=np.uint8)
            else:
                mask = np.zeros(img.shape[:2], dtype=np.uint8)

            mask = ((mask > 0).astype(np.uint8) * 255)

            img, mask = smart_crop_dual(img, mask, context=1.2)
            img = normalizer(img)

            img = cv2.resize(img, (Config.SAVE_SIZE, Config.SAVE_SIZE), interpolation=cv2.INTER_LANCZOS4)
            mask = cv2.resize(mask, (Config.SAVE_SIZE, Config.SAVE_SIZE), interpolation=cv2.INTER_NEAREST)

            Image.fromarray(img).save(out_rgb_p)
            Image.fromarray(mask).save(out_msk_p)

            valid_files.append(base + ".png")
        except Exception:
            continue

    return valid_files

# ---------------------------
# 4) LOAD LABELS + PREPROCESS FULL IMAGES
# ---------------------------
df_train_raw = pd.read_csv(Config.TRAIN_LABELS_CSV)
if "image_name" not in df_train_raw.columns:
    df_train_raw.rename(columns={"sample_index": "image_name", "filename": "image_name"}, inplace=True)

le = LabelEncoder()
df_train_raw["label_encoded"] = le.fit_transform(df_train_raw["label"])
NUM_CLASSES = len(le.classes_)
print("Classes:", list(le.classes_))

valid_train = preprocess_rgb_and_mask(
    df_train_raw, Config.TRAIN_IMG_DIR, Config.TRAIN_MASK_DIR,
    Config.FAST_TRAIN_IMG, Config.FAST_TRAIN_MASK, is_test=False
)
df_train_raw["processed_filename"] = df_train_raw["image_name"].astype(str).apply(lambda x: x.split(".")[0] + ".png")
df_all = df_train_raw[df_train_raw["processed_filename"].isin(valid_train)].reset_index(drop=True)
print("Train slides:", len(df_all))

test_files = os.listdir(Config.TEST_IMG_DIR)
df_test_raw = pd.DataFrame({"filename": test_files})
valid_test = preprocess_rgb_and_mask(
    df_test_raw, Config.TEST_IMG_DIR, Config.TEST_MASK_DIR,
    Config.FAST_TEST_IMG, Config.FAST_TEST_MASK, is_test=True
)
df_test_raw["processed_filename"] = df_test_raw["filename"].astype(str).apply(lambda x: x.split(".")[0] + ".png")
df_test = df_test_raw[df_test_raw["processed_filename"].isin(valid_test)].reset_index(drop=True)
print("Test slides:", len(df_test))

# ---------------------------
# 5) TILE EXTRACTION (mask guided, cached to disk)
# ---------------------------
def safe_crop_np(img, x0, y0, size, is_mask=False):
    h, w = img.shape[:2]
    x1, y1 = x0 + size, y0 + size

    pad_l = max(0, -x0)
    pad_t = max(0, -y0)
    pad_r = max(0, x1 - w)
    pad_b = max(0, y1 - h)

    x0c, y0c = max(0, x0), max(0, y0)
    x1c, y1c = min(w, x1), min(h, y1)

    crop = img[y0c:y1c, x0c:x1c]
    if pad_l or pad_t or pad_r or pad_b:
        if is_mask:
            crop = cv2.copyMakeBorder(crop, pad_t, pad_b, pad_l, pad_r, cv2.BORDER_CONSTANT, value=0)
        else:
            crop = cv2.copyMakeBorder(crop, pad_t, pad_b, pad_l, pad_r, cv2.BORDER_REFLECT_101)

    if crop.shape[0] != size or crop.shape[1] != size:
        interp = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
        crop = cv2.resize(crop, (size, size), interpolation=interp)
    return crop

def mask_fraction(tile_mask):
    return float((tile_mask > 0).mean()) if tile_mask.size else 0.0

def get_component_centers(mask):
    # mask uint8 [0,255]
    m = (mask > 0).astype(np.uint8)
    num, labels, stats, centroids = cv2.connectedComponentsWithStats(m, connectivity=8)
    centers = []
    weights = []
    for i in range(1, num):
        area = stats[i, cv2.CC_STAT_AREA]
        if area < 10:
            continue
        cx, cy = centroids[i]
        centers.append((int(cx), int(cy)))
        weights.append(area)
    if len(centers) == 0:
        return None, None
    weights = np.array(weights, dtype=np.float64)
    weights = weights / (weights.sum() + 1e-9)
    return centers, weights

def extract_tiles_for_split(df, rgb_dir, mask_dir, out_dir, is_test=False):
    os.makedirs(out_dir, exist_ok=True)

    print(f"Extracting cached tiles to {out_dir} ...")
    for _, row in tqdm(df.iterrows(), total=len(df)):
        fname = row["processed_filename"]
        slide_id = fname.split(".")[0]

        slide_out = os.path.join(out_dir, slide_id)
        if os.path.exists(slide_out):
            # already extracted
            continue
        os.makedirs(slide_out, exist_ok=True)

        rgb_path = os.path.join(rgb_dir, fname)
        msk_path = os.path.join(mask_dir, fname)

        rgb = cv2.imread(rgb_path)
        if rgb is None:
            continue
        rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)

        msk = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)
        if msk is None:
            msk = np.zeros((rgb.shape[0], rgb.shape[1]), dtype=np.uint8)

        centers, weights = get_component_centers(msk)
        rng = np.random.default_rng(Config.SEED + (hash(slide_id) % 10_000_000))

        H, W = msk.shape[:2]
        ts = Config.TILE_SIZE

        saved = 0
        tries = 0

        # add 1 "global" tile first (center)
        cx0, cy0 = W // 2, H // 2
        x0, y0 = cx0 - ts // 2, cy0 - ts // 2
        g = safe_crop_np(rgb, x0, y0, ts, is_mask=False)
        Image.fromarray(g).save(os.path.join(slide_out, f"tile_{saved:03d}.jpg"), quality=95)
        saved += 1

        while saved < Config.MAX_TILES_PER_SLIDE and tries < Config.MAX_TRIES:
            tries += 1

            # choose center from components if possible, otherwise uniform random
            if centers is not None and rng.random() < 0.85:
                idx = int(rng.choice(len(centers), p=weights))
                cx, cy = centers[idx]
                # jitter so tiles cover surrounding context
                cx = int(np.clip(cx + rng.integers(-ts//3, ts//3 + 1), 0, W - 1))
                cy = int(np.clip(cy + rng.integers(-ts//3, ts//3 + 1), 0, H - 1))
            else:
                cx = int(rng.integers(0, W))
                cy = int(rng.integers(0, H))

            x0 = cx - ts // 2
            y0 = cy - ts // 2

            tmask = safe_crop_np(msk, x0, y0, ts, is_mask=True)
            if mask_fraction(tmask) < Config.MIN_MASK_FRAC:
                continue

            timg = safe_crop_np(rgb, x0, y0, ts, is_mask=False)
            Image.fromarray(timg).save(os.path.join(slide_out, f"tile_{saved:03d}.jpg"), quality=95)
            saved += 1

        # if too few tiles, duplicate last
        if saved == 0:
            continue
        while saved < Config.MAX_TILES_PER_SLIDE:
            src = os.path.join(slide_out, f"tile_{saved-1:03d}.jpg")
            dst = os.path.join(slide_out, f"tile_{saved:03d}.jpg")
            shutil.copy(src, dst)
            saved += 1

# extract tiles for train/test
extract_tiles_for_split(df_all, Config.FAST_TRAIN_IMG, Config.FAST_TRAIN_MASK, Config.TILES_TRAIN_DIR, is_test=False)
extract_tiles_for_split(df_test, Config.FAST_TEST_IMG, Config.FAST_TEST_MASK, Config.TILES_TEST_DIR, is_test=True)

# build mapping slide_id -> list(tile_paths)
def build_tiles_index(df, tiles_root):
    mapping = {}
    for _, row in df.iterrows():
        slide_id = row["processed_filename"].split(".")[0]
        folder = os.path.join(tiles_root, slide_id)
        if not os.path.isdir(folder):
            continue
        paths = [os.path.join(folder, f) for f in sorted(os.listdir(folder)) if f.endswith(".jpg")]
        if len(paths) == 0:
            continue
        mapping[slide_id] = paths
    return mapping

train_tiles_map = build_tiles_index(df_all, Config.TILES_TRAIN_DIR)
test_tiles_map = build_tiles_index(df_test, Config.TILES_TEST_DIR)

# keep only slides that actually have tiles
df_all = df_all[df_all["processed_filename"].apply(lambda x: x.split(".")[0] in train_tiles_map)].reset_index(drop=True)
df_test = df_test[df_test["processed_filename"].apply(lambda x: x.split(".")[0] in test_tiles_map)].reset_index(drop=True)

# ---------------------------
# 6) TRANSFORMS (ViT mean/std, simple and fast)
# ---------------------------
MEAN = [0.5, 0.5, 0.5]
STD  = [0.5, 0.5, 0.5]

train_tf = transforms.Compose([
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

val_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

# ---------------------------
# 7) DATASET (bag of tiles per slide)
# ---------------------------
class BagDataset(Dataset):
    def __init__(self, df, tiles_map, train=True, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tiles_map = tiles_map
        self.train = train
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        slide_id = row["processed_filename"].split(".")[0]
        tile_paths = self.tiles_map[slide_id]

        K = Config.K_TRAIN if self.train else Config.K_VAL

        rng = np.random.default_rng(Config.SEED + idx if not self.train else None)

        if self.train:
            chosen = rng.choice(len(tile_paths), size=K, replace=(len(tile_paths) < K))
        else:
            # deterministic subset for val/test
            chosen = np.linspace(0, len(tile_paths) - 1, K).round().astype(int) if len(tile_paths) >= K else np.arange(K) % len(tile_paths)

        tiles = []
        for j in chosen:
            p = tile_paths[int(j)]
            img = Image.open(p).convert("RGB")
            img = train_tf(img) if self.train else val_tf(img)
            tiles.append(img)

        x = torch.stack(tiles, dim=0)  # (K,3,H,W)

        if self.is_test:
            return x, row["filename"]
        y = torch.tensor(int(row["label_encoded"]), dtype=torch.long)
        return x, y

def collate_bags(batch):
    xs, ys = zip(*batch)
    x = torch.stack(xs, dim=0)  # (B,K,3,H,W)
    if isinstance(ys[0], torch.Tensor):
        y = torch.stack(ys, dim=0)
        return x, y
    return x, list(ys)

# ---------------------------
# 8) MODEL: Phikon encoder + attention MIL pooling
# ---------------------------
class AttnMIL(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.encoder = ViTModel.from_pretrained(Config.BACKBONE)

        d = self.encoder.config.hidden_size

        self.attn = nn.Sequential(
            nn.Linear(d, d // 2),
            nn.Tanh(),
            nn.Linear(d // 2, 1)
        )
        self.cls = nn.Linear(d, num_classes)

    def forward(self, x):
        # x: (B,K,3,H,W)
        B, K, C, H, W = x.shape
        x = x.view(B * K, C, H, W)

        out = self.encoder(pixel_values=x, interpolate_pos_encoding=True)
        feat = out.last_hidden_state[:, 0, :]  # CLS token (B*K, D)
        feat = feat.view(B, K, -1)             # (B,K,D)

        a = self.attn(feat)                    # (B,K,1)
        a = torch.softmax(a, dim=1)            # weights over tiles
        pooled = (a * feat).sum(dim=1)         # (B,D)

        logits = self.cls(pooled)              # (B,C)
        return logits

def set_backbone_trainable(model, trainable: bool):
    for p in model.encoder.parameters():
        p.requires_grad = trainable

# ---------------------------
# 9) TRAINING
# ---------------------------
def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro")

def train_one_epoch(model, loader, optimizer, scaler, criterion):
    model.train()
    losses = []
    all_p, all_t = [], []

    for x, y in loader:
        x = x.to(Config.DEVICE, non_blocking=True)
        y = y.to(Config.DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", enabled=(Config.DEVICE.type == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        losses.append(float(loss.item()))
        all_p.append(logits.argmax(1).detach().cpu().numpy())
        all_t.append(y.detach().cpu().numpy())

    all_p = np.concatenate(all_p)
    all_t = np.concatenate(all_t)
    return float(np.mean(losses)), float(macro_f1(all_t, all_p))

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    losses = []
    all_p, all_t = [], []

    for x, y in loader:
        x = x.to(Config.DEVICE, non_blocking=True)
        y = y.to(Config.DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=(Config.DEVICE.type == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        losses.append(float(loss.item()))
        all_p.append(logits.argmax(1).cpu().numpy())
        all_t.append(y.cpu().numpy())

    all_p = np.concatenate(all_p)
    all_t = np.concatenate(all_t)
    return float(np.mean(losses)), float(macro_f1(all_t, all_p))

def build_criterion(train_labels):
    # class weights help macro F1 when classes are imbalanced
    counts = np.bincount(train_labels, minlength=NUM_CLASSES).astype(np.float32)
    w = (counts.sum() / (counts + 1e-6))
    w = w / w.mean()
    w = torch.tensor(w, dtype=torch.float32, device=Config.DEVICE)
    return nn.CrossEntropyLoss(weight=w, label_smoothing=Config.LABEL_SMOOTH)

def train_fold(fold, train_idx, val_idx):
    print(f"\n================ FOLD {fold+1}/{Config.N_FOLDS} ================")

    tr_df = df_all.iloc[train_idx].reset_index(drop=True)
    va_df = df_all.iloc[val_idx].reset_index(drop=True)

    # balanced slide sampler
    class_counts = tr_df["label_encoded"].value_counts().sort_index().values
    cw = 1.0 / (class_counts + 1e-6)
    sample_weights = [cw[l] for l in tr_df["label_encoded"].values]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

    tr_ds = BagDataset(tr_df, train_tiles_map, train=True, is_test=False)
    va_ds = BagDataset(va_df, train_tiles_map, train=False, is_test=False)

    g = torch.Generator()
    g.manual_seed(Config.SEED + fold)

    tr_loader = DataLoader(
        tr_ds,
        batch_size=Config.BATCH_SIZE,
        sampler=sampler,
        num_workers=Config.NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4,
        worker_init_fn=seed_worker,
        generator=g,
        collate_fn=collate_bags,
        drop_last=True
    )
    va_loader = DataLoader(
        va_ds,
        batch_size=Config.BATCH_SIZE,
        shuffle=False,
        num_workers=Config.NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4,
        worker_init_fn=seed_worker,
        generator=g,
        collate_fn=collate_bags,
        drop_last=False
    )

    model = AttnMIL(NUM_CLASSES).to(Config.DEVICE)

    # freeze encoder for a few epochs (helps small data)
    set_backbone_trainable(model, False)

    params = [
        {"params": model.attn.parameters(), "lr": Config.LR_HEAD},
        {"params": model.cls.parameters(), "lr": Config.LR_HEAD},
        {"params": model.encoder.parameters(), "lr": Config.LR_BACKBONE},
    ]
    optimizer = optim.AdamW(params, weight_decay=Config.WEIGHT_DECAY)

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.EPOCHS, eta_min=1e-6)

    scaler = torch.amp.GradScaler("cuda", enabled=(Config.DEVICE.type == "cuda"))
    criterion = build_criterion(tr_df["label_encoded"].values)

    best_f1 = 0.0
    patience = 8
    bad = 0

    for epoch in range(Config.EPOCHS):
        if epoch == Config.FREEZE_EPOCHS:
            set_backbone_trainable(model, True)

        tr_loss, tr_f1 = train_one_epoch(model, tr_loader, optimizer, scaler, criterion)
        va_loss, va_f1 = validate(model, va_loader, criterion)
        scheduler.step()

        print(
            f"Ep {epoch+1:02d} | "
            f"train loss {tr_loss:.4f} f1 {tr_f1:.4f} | "
            f"val loss {va_loss:.4f} f1 {va_f1:.4f} | "
            f"best {best_f1:.4f}"
        )

        if va_f1 > best_f1:
            best_f1 = va_f1
            torch.save(model.state_dict(), f"fold_{fold}_best.pth")
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print("Early stopping")
                break

    del model, optimizer, scheduler, scaler, tr_loader, va_loader, tr_ds, va_ds
    torch.cuda.empty_cache()
    gc.collect()

    return best_f1

# ---------------------------
# 10) RUN CV
# ---------------------------
skf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True, random_state=Config.SEED)
fold_scores = []
for fold, (train_idx, val_idx) in enumerate(skf.split(df_all, df_all["label_encoded"])):
    fold_scores.append(train_fold(fold, train_idx, val_idx))

print("\n==================================")
print(f"FINAL CV F1: {np.mean(fold_scores):.4f}")
print("==================================")

# ---------------------------
# 11) INFERENCE (fold ensemble)
# ---------------------------
@torch.no_grad()
def predict(models_list, loader):
    all_names, all_pred = [], []

    for x, names in tqdm(loader, total=len(loader)):
        x = x.to(Config.DEVICE, non_blocking=True)

        probs_sum = torch.zeros((x.size(0), NUM_CLASSES), device=Config.DEVICE)
        for m in models_list:
            with torch.amp.autocast(device_type="cuda", enabled=(Config.DEVICE.type == "cuda")):
                logits = m(x)
                probs_sum += torch.softmax(logits, dim=1)

        probs_sum /= len(models_list)
        pred_idx = probs_sum.argmax(1).detach().cpu().numpy()

        all_pred.extend(pred_idx.tolist())
        all_names.extend(list(names))

    return all_names, all_pred

print("\nStarting inference...")
test_ds = BagDataset(df_test, test_tiles_map, train=False, is_test=True)
test_loader = DataLoader(
    test_ds,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    num_workers=Config.NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
    worker_init_fn=seed_worker,
    collate_fn=collate_bags,
    drop_last=False
)

models_list = []
for i in range(Config.N_FOLDS):
    p = f"fold_{i}_best.pth"
    if os.path.exists(p):
        m = AttnMIL(NUM_CLASSES).to(Config.DEVICE)
        m.load_state_dict(torch.load(p, map_location="cpu"))
        m.eval()
        models_list.append(m)

test_names, test_pred_idx = predict(models_list, test_loader)
test_pred_lbl = le.inverse_transform(np.array(test_pred_idx))

sub = pd.DataFrame({"sample_index": test_names, "label": test_pred_lbl})
sub.to_csv("submission.csv", index=False)
print("Submission Saved: submission.csv")

2025-12-14 11:12:02.817710: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765710723.008464      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765710723.059309      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

config.json:   0%|          | 0.00/492 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Ep 01 | train loss 1.4254 f1 0.2439 | val loss 1.6277 f1 0.1252 | best 0.0000
Ep 02 | train loss 1.2697 f1 0.3535 | val loss 1.5365 f1 0.2134 | best 0.1252
Ep 03 | train loss 1.2318 f1 0.3401 | val loss 1.5415 f1 0.2179 | best 0.2134
Ep 04 | train loss 1.2114 f1 0.4062 | val loss 1.6158 f1 0.2378 | best 0.2179
Ep 05 | train loss 1.0822 f1 0.4557 | val loss 1.6318 f1 0.2717 | best 0.2378
Ep 06 | train loss 0.9548 f1 0.5454 | val loss 1.7271 f1 0.2258 | best 0.2717
Ep 07 | train loss 0.9269 f1 0.6303 | val loss 1.6844 f1 0.2466 | best 0.2717
Ep 08 | train loss 0.7307 f1 0.7294 | val loss 1.6809 f1 0.3258 | best 0.2717
Ep 09 | train loss 0.7468 f1 0.7630 | val loss 1.8498 f1 0.2392 | best 0.3258
Ep 10 | train loss 0.6646 f1 0.8089 | val loss 1.7552 f1 0.2920 | best 0.3258
Ep 11 | train loss 0.5811 f1 0.8711 | val loss 1.8073 f1 0.3305 | best 0.3258
Ep 12 | train loss 0.5263 f1 0.9107 | val loss 1.8144 f1 0.2815 | best 0.3305
Ep 13 | train loss 0.5260 f1 0.9164 | val loss 1.8475 f1 0.2975 

  0%|          | 0/120 [00:00<?, ?it/s]

Submission Saved: submission.csv
